In [0]:
%pip install fitparse

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.triathlon_utils import (
    parse_fit, 
    clean_fit_data, 
    calculate_elapsed_time, 
    apply_triathlon_style,
    format_time_axis,
    export_for_instagram,
    KLEUREN_PALETTE
)

from utils.config import DEFAULT_ATHLETE, DEFAULT_PATHS

In [0]:
def haversine_2d(lat1, lon1, lat2, lon2):
    """Berekend de 2D afstand op een bolvormig aardoppervlak (in meters)."""
    R = 6371000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)

    a = np.sin(dphi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def calculate_3d_distances(df):
    """Voegt 2D en 3D stap-afstanden en cumulatieve afstanden toe met Semicircle correctie."""
    df = df.copy()

    # 1. Uniforme kolomnamen
    column_mapping = {
        'timestamp': 'time',
        'lat': 'latitude',
        'position_lat': 'latitude',
        'lon': 'longitude',
        'lng': 'longitude',
        'position_long': 'longitude',
        'altitude': 'elevation',
        'enhanced_altitude': 'elevation'
    }
    df = df.rename(columns=column_mapping)

    # 2. Semicircles omzetten naar graden indien van toepassing
    if df['latitude'].abs().max() > 180:
        df['latitude'] = df['latitude'] * (180.0 / (2**31))
    if df['longitude'].abs().max() > 180:
        df['longitude'] = df['longitude'] * (180.0 / (2**31))

    # 3. Sorteer op tijd
    if 'time' in df.columns:
        df = df.sort_values('time').reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    # 4. Verschuif coördinaten voor stapberekeningen
    lat1, lon1, ele1 = df['latitude'].shift(1), df['longitude'].shift(1), df['elevation'].shift(1)
    lat2, lon2, ele2 = df['latitude'], df['longitude'], df['elevation']

    # 5. Bereken 2D afstand en hoogteverschil per stap
    step_2d = haversine_2d(lat1, lon1, lat2, lon2).fillna(0)
    step_ele_diff = (ele2 - ele1).fillna(0)

    # Filter extreem grote sprongen uit (bijv. GPS-glitches of herstarts na pauze)
    step_2d = step_2d.apply(lambda x: x if x < 200 else 0)

    # 6. 3D afstand via Pythagoras (3D^2 = 2D^2 + dEle^2)
    step_3d = np.sqrt(step_2d**2 + step_ele_diff**2)

    # 7. Cumulatieve afstanden (km) en opgebouwd verschil (m)
    df['step_2d_m'] = step_2d
    df['step_3d_m'] = step_3d
    df['cum_dist_2d_km'] = step_2d.cumsum() / 1000.0
    df['cum_dist_3d_km'] = step_3d.cumsum() / 1000.0
    df['diff_3d_2d_m'] = step_3d - step_2d
    df['cum_diff_m'] = df['diff_3d_2d_m'].cumsum()

    return df

def create_elevation_factor_chart(df, stage_num=6):
    """
    Genereert de visualisatie van 2D vs 3D afstand.
    """
    df_calc = calculate_3d_distances(df.copy())
    
    # 1. Gebruik het donkere thema
    plt.style.use('dark_background')
    
    fig, (ax1, ax2) = plt.subplots(
        2, 1, 
        figsize=(12, 7.5), 
        facecolor='black',
        sharex=True, 
        gridspec_kw={'height_ratios': [2, 1]}
    )
    ax1.set_facecolor('black')
    ax2.set_facecolor('black')
    
    # Kleurenpalet
    COLOR_2D = '#00F5FF'       # Lichtgrijs
    COLOR_3D = '#00F5FF'       # Cyaan/Turquoise
    COLOR_ELEV = '#4A5568'     # Donkergrijs/Blauw
    COLOR_DIFF = '#FF2A6D'     # Felroze/Magenta
    
    # --- BOVENSTE GRAFIEK: 2D vs 3D + Hoogteprofiel ---
    ax1_elev = ax1.twinx()
    
    min_elev = df_calc['elevation'].min()
    max_elev = df_calc['elevation'].max()
    ax1_elev.set_ylim(min_elev - 150, max_elev + 200)
    
    # Schaduw-invulling voor hoogteprofiel op de achtergrond
    ax1_elev.fill_between(
        df_calc['cum_dist_2d_km'], 
        df_calc['elevation'], 
        y2=min_elev - 150,
        color=COLOR_ELEV, 
        alpha=0.35, 
        label='Hoogteprofiel'
    )
    ax1_elev.plot(df_calc['cum_dist_2d_km'], df_calc['elevation'], color='#718096', alpha=0.5, lw=1)
    ax1_elev.set_ylabel('Hoogte (m)', color='#A0A0A0', fontsize=10, fontweight='bold')
    ax1_elev.tick_params(axis='y', colors='#A0A0A0')
    ax1_elev.grid(False)  # <--- Raster uit op hoogte-as

    # Lijnen voor 2D vs 3D afstandsopbouw
    ax1.plot(df_calc['cum_dist_2d_km'], df_calc['cum_dist_2d_km'], color=COLOR_2D, linestyle='--', lw=2, label='2D Afstand (Platte kaart)')
    ax1.plot(df_calc['cum_dist_2d_km'], df_calc['cum_dist_3d_km'], color=COLOR_3D, lw=2.5, label='3D Afstand (Werkelijk)')
    
    ax1.set_ylabel('Berekende Afstand (km)', color='white', fontsize=10, fontweight='bold')
    ax1.set_title(
        f"ETAPPE {stage_num} PERFORMANCE: THE ELEVATION FACTOR (2D PLAT VS. 3D WERKELIJK)", 
        color='white',
        fontsize=12, 
        fontweight='bold', 
        pad=15
    )
    ax1.tick_params(axis='x', colors='white')
    ax1.tick_params(axis='y', colors='white')
    ax1.grid(False)  # <--- Raster uit op bovenste hoofdas
    
    # Stats Box
    total_2d = df_calc['cum_dist_2d_km'].iloc[-1]
    total_3d = df_calc['cum_dist_3d_km'].iloc[-1]
    diff_m = df_calc['cum_diff_m'].iloc[-1]
    pct_extra = (diff_m / (total_2d * 1000)) * 100
    
    stats_text = (
        f"Totale 2D Afstand: {total_2d:.2f} km\n"
        f"Totale 3D Afstand: {total_3d:.2f} km\n"
        f"---------------------------\n"
        f"Extra Steilte-meters: +{diff_m:.0f} m (+{pct_extra:.1f}%)"
    )

    ax1.text(
        0.02, 0.95, stats_text, 
        transform=ax1.transAxes, 
        fontsize=10, color='white', fontweight='bold',
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#111827', alpha=0.85, edgecolor='#374151')
    )
    
    ax1.legend(loc='lower right', facecolor='#121212', edgecolor='#333333', labelcolor='white')

    # --- ONDERSTE GRAFIEK: Cumulatieve extra meters ---
    ax2.plot(df_calc['cum_dist_2d_km'], df_calc['cum_diff_m'], color=COLOR_DIFF, lw=2, label='Cumulatieve extra 3D-meters')
    ax2.fill_between(df_calc['cum_dist_2d_km'], 0, df_calc['cum_diff_m'], color=COLOR_DIFF, alpha=0.25)
    
    ax2.set_xlabel('Kaartafstand (km)', color='white', fontsize=10, fontweight='bold')
    ax2.set_ylabel('Extra meters (m)', color='white', fontsize=10, fontweight='bold')
    ax2.tick_params(axis='x', colors='white')
    ax2.tick_params(axis='y', colors='white')
    ax2.grid(False)  # <--- Raster uit op onderste hoofdas
    
    ax2.legend(loc='lower right', facecolor='#121212', edgecolor='#333333', labelcolor='white')

    plt.tight_layout()
    return fig, df_calc

In [0]:
# Het FIT-bestand van etappe 6
fit_file_path = f"{DEFAULT_PATHS.fit_dir}/20260917_stubaier_hohenweg_etappe_6.fit"

# 1. Inlezen & Opschonen via de utils
df = parse_fit(fit_file_path)
df_clean = clean_fit_data(df)

# 2. Grafiek genereren
fig, df_processed = create_elevation_factor_chart(
    df_clean,
    stage_num=6
)

# 3. Exporteer naar landscape formaat voor Instagram
export_for_instagram(fig, filename="20260917_elevation_factor", format_type='landscape')